# Clase 12 · Predecir un sí o un no

**Estadística Descriptiva e Inferencial** · Módulo 4 · Sesión 12 de 14

---

## De predecir números a predecir etiquetas

En la Clase 11 predijimos **cuántos días** tardaba un caso. Hoy predecimos algo distinto:

> ¿Este cliente entrará en mora? ¿Esta transacción es fraude? ¿Este pasajero sobrevivió?

La respuesta ya no es un número en una escala: es **sí o no**. Y eso cambia dos cosas:

| | Clase 11 · lineal | Clase 12 · logística |
|---|---|---|
| Lo que predice | un número (días) | una **probabilidad** de sí |
| El modelo | una recta | una **curva en S** |
| Se valida con | RMSE, R² | **matriz de confusión, AUC** |

Este es el modelo que está detrás de **casi todo scorecard de crédito**.

## Los datos de hoy

El **Titanic**: 891 pasajeros, y sabemos cuáles sobrevivieron. Es el dataset clásico para
aprender clasificación, y lo vas a volver a ver en el proyecto final — así que conviene
conocerlo bien.

## El laboratorio

| Bloque | Min | Qué haces |
|---|---|---|
| 1 | 12 | Ves fallar a la regresión lineal y aparece la curva en S |
| 2 | 13 | Interpretas los coeficientes como **odds ratio** |
| 3 | 12 | Construyes la matriz de confusión |
| 4 | 12 | **Mueves el umbral y ves cambiar toda la decisión** |
| 5 | 8 | La curva ROC y el AUC |
| Apéndice | 5 | Tres clases en vez de dos, con iris |

---
## Celda 0 · Preparación y datos

Los datos se leen directamente del repo del curso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_curve, roc_auc_score)
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

URL = ("https://raw.githubusercontent.com/josefrodrim/"
       "Estad-stica-Descriptiva-E-Inferencial/main/"
       "Proyecto_final_titanic/Data/Titanic-Dataset.csv")

def cargar():
    try:
        d = pd.read_csv(URL)
        print("Datos cargados desde el repo del curso.")
        return d
    except Exception as e:
        print(f"No se pudo descargar ({type(e).__name__}).")
        print("Sube Titanic-Dataset.csv con el boton de archivos de Colab.")
        return pd.read_csv("Titanic-Dataset.csv")

titanic = cargar()
print(f"\n{len(titanic)} pasajeros, {len(titanic.columns)} columnas")
print(titanic[["Survived", "Pclass", "Sex", "Age", "Fare"]].head(5).to_string(index=False))

### Las columnas que vamos a usar

| Columna | Qué es |
|---|---|
| `Survived` | **1 si sobrevivió, 0 si no.** Es lo que queremos predecir |
| `Sex` | male / female |
| `Pclass` | clase del billete: 1, 2 o 3 |
| `Age` | edad (tiene 177 valores faltantes) |

Preparamos dos cosas: convertir el sexo a número y rellenar las edades que faltan.

In [ ]:
# ── DEMOSTRACIÓN: preparación mínima ─────────────────────────────────────
datos = titanic.copy()
datos["mujer"] = (datos["Sex"] == "female").astype(int)   # 1 = mujer, 0 = hombre
datos["edad"] = datos["Age"].fillna(datos["Age"].median())

print(f"Edades faltantes: {titanic['Age'].isna().sum()}")
print(f"Se rellenaron con la mediana: {titanic['Age'].median():.1f} anos")
print()
print("OJO: rellenar con la mediana es la solucion mas simple, no la mejor.")
print("En el proyecto final vas a tener que pensar mejor que hacer con esos 177.")
print()
print("=" * 56)
print(f"Sobrevivio el {100*datos.Survived.mean():.2f} % de los pasajeros")
print("=" * 56)
print()
print("Tasas de supervivencia por grupo:")
for c in ["Sex", "Pclass"]:
    for k, v in datos.groupby(c)["Survived"].mean().items():
        print(f"  {c} = {str(k):8}: {100*v:5.1f} %   (n = {(datos[c]==k).sum()})")

---
# Bloque 1 · Por qué hace falta un modelo nuevo  ·  12 min

La pregunta obvia: ¿por qué no usamos la regresión lineal de la Clase 11, poniendo
`Survived` (que vale 0 o 1) como variable a predecir?

Vamos a intentarlo y ver qué pasa.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X = datos[["mujer", "Pclass", "edad"]]
y = datos["Survived"]

lineal = None          # ajusta una LinearRegression con TODOS los datos
pred_lineal = None     # sus predicciones

print("mínimo:", pred_lineal.min(), "máximo:", pred_lineal.max())

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("predicción mínima de la lineal", pred_lineal.min(), -0.1434, tol=1e-3),
     check("predicción máxima", pred_lineal.max(), 1.0869, tol=1e-3),
     check_bool("hay predicciones fuera de [0, 1]",
                ((pred_lineal < 0) | (pred_lineal > 1)).sum() > 0)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — La solución: una curva en forma de S

La regresión logística resuelve el problema pasando el resultado por una función que
**aplasta cualquier número al rango [0, 1]**:

$$p = \frac{1}{1 + e^{-z}} \qquad \text{donde} \qquad z = a + b_1 x_1 + b_2 x_2 + \dots$$

La parte de `z` es exactamente la recta de la Clase 11. Lo nuevo es la función que la
envuelve, y que se llama **sigmoide** o **función logística**.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def sigmoide(z):
    """Convierte cualquier número real en una probabilidad entre 0 y 1."""
    return None

for z in [-4, -2, 0, 2, 4]:
    print(f"z = {z:>3}  ->  p = {sigmoide(z)}")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("sigmoide(0)", sigmoide(0), 0.5),
     check("sigmoide(2)", sigmoide(2), 0.8808, tol=1e-3),
     check("sigmoide(-2)", sigmoide(-2), 0.1192, tol=1e-3),
     check_bool("nunca sale del rango, ni con números enormes",
                0 < sigmoide(-500) and sigmoide(500) < 1.0000001)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

---
# Bloque 2 · El modelo, y cómo se leen sus coeficientes  ·  13 min

Ajustar el modelo es igual de fácil que antes. Lo distinto es **interpretarlo**.

Antes de eso hace falta una idea: las **odds** (en español, «momios» o «probabilidades en
contra», pero casi todo el mundo dice *odds*).

$$odds = \frac{p}{1-p}$$

| Probabilidad | Odds | Se lee |
|---|---|---|
| 0.50 | 1.0 | uno a uno |
| 0.75 | 3.0 | tres a uno a favor |
| 0.20 | 0.25 | uno a cuatro en contra |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Calcula la probabilidad y las odds de sobrevivir, por sexo.

p_mujer  = None    # proporción de mujeres que sobrevivieron
p_hombre = None

odds_mujer  = None   # p / (1 - p)
odds_hombre = None

odds_ratio = None    # odds_mujer / odds_hombre

print(f"OR = {odds_ratio}")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("probabilidad de sobrevivir, mujeres", p_mujer, 0.7420, tol=1e-3),
     check("probabilidad de sobrevivir, hombres", p_hombre, 0.1889, tol=1e-3),
     check("odds de las mujeres", odds_mujer, 2.8765, tol=1e-3),
     check("odds ratio mujer/hombre", odds_ratio, 12.3507, tol=1e-2)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — Ahora el modelo completo

Los coeficientes de una regresión logística están **en escala de log-odds**, que no se
puede interpretar directamente. Para leerlos hay que aplicarles `exp()`, y entonces se
convierten en **odds ratio**.

$$OR = e^{\text{coeficiente}}$$

| OR | Significa |
|---|---|
| **> 1** | esa variable **aumenta** las odds |
| **= 1** | no cambia nada |
| **< 1** | las **reduce** |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
modelo = None       # LogisticRegression(max_iter=1000) sobre TODOS los datos

coefs = None        # modelo.coef_[0]
ors   = None        # exponencial de los coeficientes

tabla_or = pd.DataFrame({"variable": ["mujer", "Pclass", "edad"],
                         "coeficiente": None, "odds_ratio": None})
print(tabla_or)

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("odds ratio de 'mujer'", ors[0], 12.4255, tol=0.05),
     check_bool("el OR de Pclass es menor que 1 (peor clase, menos odds)", ors[1] < 1),
     check_bool("el OR de edad es menor que 1 (más edad, menos odds)", ors[2] < 1),
     check_bool("y el de mujer es mucho mayor que 1", ors[0] > 5)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · La matriz de confusión  ·  12 min

El modelo da **probabilidades**. Para tomar una decisión hay que convertirlas en un sí o
un no, y eso se hace con un **umbral**: por defecto, 0.5.

Y una vez que decides, hay **cuatro resultados posibles**:

| | Predijo NO | Predijo SÍ |
|---|---|---|
| **Realmente NO** | Verdadero negativo (VN) ✓ | Falso positivo (FP) ✗ |
| **Realmente SÍ** | Falso negativo (FN) ✗ | Verdadero positivo (VP) ✓ |

Los dos errores **no son iguales**, y en tu trabajo casi nunca cuestan lo mismo.

### Ejercicio 3.1 — Separa los datos y entrena

Como en la Clase 11. Con una novedad: `stratify=y` mantiene la misma proporción de
sobrevivientes en los dos grupos.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = None, None, None, None
# usa test_size=0.3, random_state=42, stratify=y

modelo_v = None      # entrenado SOLO con los de entrenamiento
prob_test = None     # modelo_v.predict_proba(X_test)[:, 1]

print(f"entrenamiento={len(X_train)}  prueba={len(X_test)}")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("pasajeros de entrenamiento", len(X_train), 623),
     check("pasajeros de prueba", len(X_test), 268),
     check_bool("las probabilidades están entre 0 y 1",
                prob_test.min() >= 0 and prob_test.max() <= 1),
     check_bool("stratify mantuvo la proporción",
                abs(y_train.mean() - y_test.mean()) < 0.02)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### Ejercicio 3.2 — La matriz, y las tres métricas

Ahora convertimos las probabilidades en decisiones con el umbral 0.5 y contamos los cuatro
casos.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
pred_05 = None      # 1 si la probabilidad es >= 0.5

cm = None           # confusion_matrix(y_test, pred_05)

exactitud = None    # (VP + VN) / total
precision = None    # VP / (VP + FP)
recall    = None    # VP / (VP + FN)

print(cm)

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
r = [check("verdaderos negativos", vn, 130),
     check("falsos positivos", fp, 35),
     check("falsos negativos", fn, 26),
     check("verdaderos positivos", vp, 77),
     check("exactitud", exactitud, 0.7724, tol=1e-3),
     check("precisión", precision, 0.6875, tol=1e-3),
     check("recall", recall, 0.7476, tol=1e-3)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.2")

---
# Bloque 4 · El umbral lo cambia todo  ·  12 min

**Este es el bloque más importante del día, y el que más se parece a tu trabajo.**

El 0.5 no tiene nada de sagrado. Es solo el valor por defecto.

Y moverlo cambia por completo el tipo de errores que comete el modelo. Vamos a verlo.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
filas = []
for u in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = None                  # 1 si prob_test >= u
    c = confusion_matrix(y_test, pred)
    vn_, fp_, fn_, vp_ = c.ravel()
    filas.append({"umbral": u, "VP": vp_, "FP": fp_, "FN": fn_, "VN": vn_,
                  "exactitud": None, "precision": None, "recall": None})

tabla_umbral = pd.DataFrame(filas)
print(tabla_umbral)

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("recall con umbral 0.3", tabla_umbral.recall.iloc[0], 0.8350, tol=1e-3),
     check("recall con umbral 0.7", tabla_umbral.recall.iloc[-1], 0.4854, tol=1e-3),
     check("precisión con umbral 0.3", tabla_umbral.precision.iloc[0], 0.6418, tol=1e-3),
     check("precisión con umbral 0.7", tabla_umbral.precision.iloc[-1], 0.8772, tol=1e-3),
     check_bool("al subir el umbral, el recall baja",
                tabla_umbral.recall.iloc[-1] < tabla_umbral.recall.iloc[0]),
     check_bool("y la precisión sube",
                tabla_umbral.precision.iloc[-1] > tabla_umbral.precision.iloc[0])]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — ¿Cuál umbral eliges?

Este ejercicio **no tiene respuesta numérica única**. Se te pide decidir, que es lo que
te van a pedir en el trabajo.

Piensa en dos escenarios distintos:

**Escenario A · Detección de fraude.**
Un falso negativo (fraude que se te escapa) cuesta miles de soles. Un falso positivo
(alerta en vano) cuesta 10 minutos de un analista.

**Escenario B · Bloqueo automático de tarjetas.**
Un falso positivo bloquea la tarjeta de un cliente legítimo en el supermercado, que
llama furioso y quizá se va del banco.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Para cada escenario, elige un umbral alto o bajo y explica por qué.

umbral_fraude = None       # ¿alto o bajo? pon un número
razon_fraude  = ""

umbral_bloqueo = None
razon_bloqueo  = ""

print(f"Fraude: {umbral_fraude} — {razon_fraude}")
print(f"Bloqueo: {umbral_bloqueo} — {razon_bloqueo}")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check_bool("elegiste un umbral BAJO para fraude",
                umbral_fraude is not None and umbral_fraude < 0.5,
                "si un falso negativo cuesta mucho más, conviene bajar el umbral"),
     check_bool("y un umbral ALTO para el bloqueo automático",
                umbral_bloqueo is not None and umbral_bloqueo > 0.5),
     check_bool("escribiste las dos razones",
                len(razon_fraude) > 20 and len(razon_bloqueo) > 20)]
print()
print("Bloque 4 COMPLETO" if all(r) else "Completa las dos respuestas de 4.2")

---
# Bloque 5 · La curva ROC y el AUC  ·  8 min

Si el umbral se puede mover, ¿cómo evalúas el modelo **sin depender de un umbral
concreto**?

La respuesta es la **curva ROC**: se prueban todos los umbrales posibles y se dibuja el
resultado. Y el **AUC** —el área bajo esa curva— resume todo en un número.

| AUC | Qué significa |
|---|---|
| 0.5 | el modelo no discrimina: es como lanzar una moneda |
| 0.7 – 0.8 | aceptable |
| 0.8 – 0.9 | bueno |
| > 0.9 | excelente (y sospechoso: revisa si hay fuga de información) |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
fpr, tpr, umbrales = None, None, None   # roc_curve(y_test, prob_test)

auc = None                              # roc_auc_score(y_test, prob_test)

print(f"AUC = {auc}")

In [ ]:
# ── VERIFICACIÓN 5 ──────────────────────────────────────────────────────
r = [check("AUC en prueba", auc, 0.8375, tol=1e-3),
     check_bool("el AUC es mayor que 0.5 (mejor que el azar)", auc > 0.5),
     check_bool("y cae en el rango 'bueno'", 0.8 <= auc < 0.9)]
print()
print("REPORTE COMPLETO, como se escribiria:")
print()
print(f"  'Modelo de regresion logistica con tres variables sobre {len(X)} pasajeros")
print(f"   ({len(X_train)} entrenamiento, {len(X_test)} prueba). AUC en prueba = {auc:.3f}.")
print(f"   Con umbral 0.5: exactitud {exactitud:.3f}, precision {precision:.3f}, recall {recall:.3f},")
print(f"   frente a un {1-y_test.mean():.3f} de la regla trivial. El sexo es el predictor")
print(f"   mas fuerte (OR = {ors[0]:.1f} manteniendo constantes clase y edad).")
print(f"   El umbral debe fijarse segun el costo relativo de cada tipo de error.'")
print()
print("Bloque 5 COMPLETO" if all(r) else "Revisa el bloque 5")

---
# Apéndice · ¿Y si hay más de dos categorías?  ·  5 min

Todo lo de hoy fue **binario**: sí o no. Pero a veces hay tres o más categorías, y la
regresión logística también sirve. Se llama **multiclase**.

> **Un matiz de nombre:** *multiclase* es elegir **una** etiqueta entre varias (esta flor
> es setosa, versicolor **o** virginica). *Multietiqueta* es distinto: un caso puede tener
> **varias** etiquetas a la vez (un correo puede ser «urgente» **y** «facturación»). Hoy
> vemos multiclase.

Usamos el dataset **iris**: 150 flores, cuatro medidas y tres especies.

In [ ]:
# ── DEMOSTRACIÓN: clasificación multiclase ───────────────────────────────
from sklearn.datasets import load_iris

ir = load_iris(as_frame=True).frame
ir.columns = ["sepalo_largo", "sepalo_ancho", "petalo_largo", "petalo_ancho", "cod"]
ir["especie"] = ir["cod"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

print(f"{len(ir)} flores, {ir.especie.nunique()} especies")
print(ir.groupby("especie")[["petalo_largo", "petalo_ancho"]].mean().round(2).to_string())

Xi = ir[["sepalo_largo", "sepalo_ancho", "petalo_largo", "petalo_ancho"]]
yi = ir["especie"]
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    Xi, yi, test_size=0.3, random_state=42, stratify=yi)

mi = LogisticRegression(max_iter=1000).fit(Xi_tr, yi_tr)
pi = mi.predict(Xi_te)

print(f"\nexactitud en prueba = {accuracy_score(yi_te, pi):.4f}")
print()
esp = ["setosa", "versicolor", "virginica"]
cmi = confusion_matrix(yi_te, pi, labels=esp)
print("Matriz de confusion 3x3 (filas = real, columnas = predicho):")
print(pd.DataFrame(cmi, index=esp, columns=esp).to_string())
print()
print(f"Los coeficientes ahora son una matriz {mi.coef_.shape}:")
print("  UNA fila por especie. El modelo calcula la probabilidad de cada una")
print("  y se queda con la mas alta.")
print()
print("QUE CAMBIA Y QUE NO:")
print("  Cambia: la matriz de confusion es 3x3 en vez de 2x2, y hay un")
print("          conjunto de coeficientes por clase.")
print("  NO cambia: la idea es la misma, y precision y recall se calculan")
print("             por clase (una contra el resto).")
print()
print("Fijate en la matriz: setosa se clasifica perfecta; los pocos errores")
print("estan entre versicolor y virginica, que son las dos que se parecen.")

In [ ]:
# ── VERIFICACIÓN APÉNDICE ───────────────────────────────────────────────
r = [check("exactitud en iris", accuracy_score(yi_te, pi), 0.9333, tol=1e-3),
     check_bool("setosa se clasifica sin errores", cmi[0].sum() == cmi[0, 0]),
     check_bool("los coeficientes son 3 filas x 4 columnas", mi.coef_.shape == (3, 4))]
print()
print("APENDICE COMPLETO" if all(r) else "Revisa el apéndice")

---
# Cierre

### Lo que cambia al predecir etiquetas

| | Clase 11 · lineal | Clase 12 · logística |
|---|---|---|
| Predice | un número | una **probabilidad** |
| La forma | una recta | una **curva en S** |
| Los coeficientes | cambio directo en y | **odds ratio**, con `exp()` |
| Se valida con | RMSE, R² | **matriz de confusión, AUC** |
| Hay que decidir | — | **el umbral** |

### Checklist de salida

- [ ] Sé por qué la regresión lineal no sirve para predecir sí/no.
- [ ] Sé leer un coeficiente como odds ratio, con `exp()`.
- [ ] Sé que odds ratio **no es** «veces más probable».
- [ ] Sé construir e interpretar una matriz de confusión.
- [ ] Sé que precisión y recall se mueven en direcciones opuestas.
- [ ] Sé que el umbral es una decisión de **negocio**, no estadística.
- [ ] Comparo siempre mi exactitud contra la regla trivial.

### Los números del día

| | |
|---|---|
| Supervivencia general | 38.4 % |
| Mujeres / hombres | 74.2 % / 18.9 % |
| **Odds ratio del sexo** | **12.4** (controlando clase y edad) |
| Predicciones lineales fuera de [0,1] | 29 de 891 |
| **AUC en prueba** | **0.838** |
| Exactitud con umbral 0.5 | 0.772 (trivial: 0.616) |
| Recall, umbral 0.3 → 0.7 | **0.835 → 0.485** |
| Precisión, lo mismo | **0.642 → 0.877** |

### Los tres errores que evita esta clase

**1 · Reportar solo la exactitud.** Si el 95 % de las transacciones son legítimas, un
modelo que diga «legítima» siempre acierta el 95 %. Y es inútil.

**2 · Confundir odds ratio con probabilidad.** Un OR de 12 no significa 12 veces más
probable: la probabilidad solo era 3.9 veces mayor (74 % contra 19 %).

**3 · Aceptar el umbral 0.5 sin pensarlo.** Es un valor por defecto, no una decisión. La
decisión depende de cuánto cuesta cada tipo de error.

### Reto para el proyecto final

El proyecto final es con **estos mismos datos**, pero completo. Hoy usamos tres variables y
rellenamos las edades con la mediana. En el proyecto vas a tener que:

- decidir qué hacer de verdad con los 177 valores faltantes de `Age`
- construir variables nuevas a partir de `Name`, `Cabin` y `Ticket`
- comparar varios modelos y justificar cuál eliges
- fijar un umbral con un criterio explícito
- escribir el informe completo

Hoy aprendiste el método. En el proyecto lo aplicas solo.

---
*Estadística Descriptiva e Inferencial · Módulo 4 · Clase 12*